# US 3.6 -- Multi-Site Generalization

A CycleGAN trained on AOI/USM images from one manufacturing site may not
generalise well to other sites.  Equipment differences, lighting conditions,
and part geometries vary between sites, causing domain shift **within** each
modality.

This notebook tests the trained CycleGAN on images from multiple sites and
evaluates cross-site translation quality.

## What you will learn

1. How site-specific differences affect translation quality
2. Evaluating translation on unseen sites
3. Strategies for improving multi-site generalization
4. The `udm-epic3 validate` CLI command

## Prerequisites

- Python 3.12, PyTorch
- Install: `uv pip install -e ".[epic3]"`
- Trained CycleGAN checkpoint
- Multi-site data (AOI and USM from 2+ sites)

---
## 1. Manufacturing Sites

We have data from multiple manufacturing sites:

| Site | AOI images | USM images | Used for |
|------|-----------|-----------|----------|
| Warstein (DE) | 800 | 600 | Training |
| Malaysia (MY) | 500 | 400 | Validation |
| Mexico (MX) | 300 | 250 | Validation |
| China (CN) | 400 | 350 | Validation |

The CycleGAN is trained on Warstein data only.  We test whether it can
translate images from other sites without retraining.

### Why sites differ

- **Equipment:** different camera models, sensor types, calibration
- **Lighting:** factory lighting conditions vary
- **Parts:** different product lines with different geometries
- **Operators:** different acquisition protocols

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from udm_epic3.models.cyclegan import CycleGANModel
from udm_epic3.translation.translate import translate_directory
from udm_epic3.evaluation.quality_metrics import compute_ssim, compute_defect_dice, evaluate_translation

---
## 2. Cross-Site Translation

We apply the Warstein-trained CycleGAN to images from each validation site
and measure translation quality.

In [ ]:
# NOTE: In practice, load trained model and real multi-site data.
# This shows the evaluation pattern and expected results.

# Simulated cross-site evaluation results
site_results = {
    "Warstein (train)": {"ssim": 0.72, "defect_dice": 0.85},
    "Malaysia":         {"ssim": 0.61, "defect_dice": 0.78},
    "Mexico":           {"ssim": 0.58, "defect_dice": 0.74},
    "China":            {"ssim": 0.55, "defect_dice": 0.71},
}

print("Cross-site translation quality:")
print(f"{'Site':<25} {'SSIM':>8} {'Defect Dice':>14}")
print("-" * 50)
for site, metrics in site_results.items():
    print(f"{site:<25} {metrics['ssim']:>8.3f} {metrics['defect_dice']:>14.3f}")

In [ ]:
# Visualise cross-site performance
sites = list(site_results.keys())
ssim_scores = [r["ssim"] for r in site_results.values()]
dice_scores = [r["defect_dice"] for r in site_results.values()]

x = np.arange(len(sites))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, ssim_scores, width, label="SSIM", color="#2196F3")
bars2 = ax.bar(x + width/2, dice_scores, width, label="Defect Dice", color="#FF9800")

ax.set_ylabel("Score", fontsize=12)
ax.set_title("Cross-Site Translation Quality", fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(sites, fontsize=10)
ax.legend()
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3, axis="y")

# Highlight training site
ax.axvspan(-0.5, 0.5, alpha=0.1, color="green", label="Training site")

for bar in bars1 + bars2:
    height = bar.get_height()
    ax.annotate(f"{height:.2f}",
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3), textcoords="offset points",
                ha="center", va="bottom", fontsize=9)

fig.tight_layout()
plt.show()

---
## 3. Understanding Quality Degradation

Translation quality drops on unseen sites for predictable reasons:

| Factor | Effect | Mitigation |
|--------|--------|------------|
| Different contrast/brightness | Generator produces wrong intensity range | Add site-specific normalization |
| Different noise patterns | Generator hallucinates noise or smooths real noise | Include multi-site data in training |
| Different part geometry | Structural features confuse the generator | Train with diverse part types |
| Different defect appearance | Defects look different across sites | Use defect-aware loss (US 3.3) |

The most effective mitigation is to **include data from multiple sites** in
the training set, even if it means mixing sites in the unpaired datasets.

---
## 4. Strategies for Multi-Site Generalization

### 4.1 Multi-site training data

Include images from all sites in `trainA` and `trainB`.  The CycleGAN
learns to handle the full range of site-specific variations.

### 4.2 Site-conditional generation

Add a site label as input to the generator so it can adapt its output
style per site.  This is more complex but can improve per-site quality.

### 4.3 Fine-tuning on new sites

Start from the Warstein-trained checkpoint and fine-tune on a small number
of images from the new site (transfer learning for CycleGAN).

### 4.4 Test-time normalization

Apply histogram matching or other normalization to make new-site images
look more like the training site before running the generator.

In [ ]:
# Comparison: single-site vs multi-site training
comparison = {
    "Warstein-only training": {
        "Warstein": 0.72, "Malaysia": 0.61, "Mexico": 0.58, "China": 0.55
    },
    "Multi-site training": {
        "Warstein": 0.70, "Malaysia": 0.68, "Mexico": 0.66, "China": 0.64
    },
}

print("SSIM comparison: single-site vs multi-site training")
print(f"{'Strategy':<30} {'Warstein':>10} {'Malaysia':>10} {'Mexico':>10} {'China':>10}")
print("-" * 75)
for strategy, scores in comparison.items():
    vals = [f"{v:>10.3f}" for v in scores.values()]
    print(f"{strategy:<30} {''.join(vals)}")

print(f"\nMulti-site training slightly reduces Warstein performance")
print(f"but significantly improves generalization to other sites.")

---
## 5. CLI: `udm-epic3 validate`

Run cross-site validation from the command line:

```bash
# Validate translation quality across all sites
udm-epic3 validate \
    --config configs/epic3_cyclegan.yaml \
    --checkpoint outputs/epic3/checkpoints/epoch_200.pt \
    --sites warstein malaysia mexico china
```

This will:
1. Load paired evaluation data for each site
2. Translate AOI images using the trained generator
3. Compute SSIM and Defect Dice per site
4. Generate a cross-site comparison report
5. Save per-site visual comparisons

The config specifies per-site data paths:

```yaml
validation:
  sites:
    warstein:
      paired_dir: data/epic3/warstein/paired
    malaysia:
      paired_dir: data/epic3/malaysia/paired
    mexico:
      paired_dir: data/epic3/mexico/paired
    china:
      paired_dir: data/epic3/china/paired
```

---
## Summary

This concludes Epic 3 -- CycleGAN Cross-Modality Translation.  Here is what
we covered across all notebooks:

| Notebook | Topic | Key takeaway |
|----------|-------|-------------|
| `epic3_00_overview` | Architecture | CycleGAN with 2 generators, 2 discriminators |
| `epic3_01_data_prep` | Data | Unpaired loading, paired evaluation |
| `epic3_02_training` | Training | Adversarial + cycle + identity losses, image pool |
| `epic3_03_defect_aware` | Enhancement | Defect preservation loss |
| `epic3_04_evaluation` | Metrics | SSIM and Defect Dice |
| `epic3_05_downstream` | Application | Train segmentation on translated data |
| `epic3_06_multisite` | Generalization | Cross-site validation and strategies |